# longer runs (reruns)

**How to use this notebook:** each person opens their own copy in their own Colab session, with **Runtime > Change runtime type > T4 GPU**, and runs **one job only** (A, B or C below). A fresh Colab session forgets everything, so every job starts with its own setup cell.

| Job | What | Approx. time (from our first runs) |
|---|---|---|
| A | DiffMM, 30 epochs, one gap setting (change `RATIO`) | about 25 to 30 min |
| B | HSTGMatch with its default epochs (50 pre-train + 100 train) | about 40 min |

Keep the browser tab open while a job runs. Everything is copied to Google Drive at the end of each job.

## Job B: HSTGMatch, default training length

Before running: upload **porto_to_hstgmatch.py** using the file panel (folder icon on the left).

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cm4604_cp3
!nvidia-smi | head -12
!git clone https://github.com/decisionintelligence/DiffMM.git
!git clone https://github.com/Nerooo-g/HSTGMatch.git
!pip install -q torch_geometric pyyaml scikit-learn
!python porto_to_hstgmatch.py --porto DiffMM/data/porto --out HSTGMatch/data/porto_r0.1 --keep_ratio 0.1 --n_train 6000 --n_extra 2000

Mounted at /content/drive
Wed Nov 11 20:30:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
%%time
# stage 1: pre-training, default 50 epochs (about 8 minutes)
!cd HSTGMatch && python scripts/pretrain.py --config configs/default.yaml --data_dir data/porto_r0.1 --batch_size 32 2>&1 | tee ../hstg_pretrain_full.txt

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(
Using device: cuda
Loading data from data/porto_r0.1...
  Grid size: 54 × 114 = 6156 cells
  Processed 8000 trajectories.
  ATA-Graph: 59528 edges over 6156 nodes
  Train: 7200  Val: 800
Model parameters: 2,474,766

Starting pre-training for 50 epochs...
Epoch    1/50  train_loss=4.4779  val_loss=4.1205  lr=9.99e-05
Epoch    2/50  train_loss=4.0654  val_loss=4.0271  lr=9.96e-05
Epoch    3/50  train_loss=3.9812  val_loss=3.9719  lr=9.91e-05
Epoch    4/50  train_loss=3.9584  val_loss=3.9989  lr=9.84e-0

In [3]:
%%time
# stage 2: supervised training, default 100 epochs (about 30 minutes)
!cd HSTGMatch && python scripts/train.py --config configs/default.yaml --data_dir data/porto_r0.1 --pretrained_encoder checkpoints/pretrain/best_encoder.pt --batch_size 32 2>&1 | tee ../hstg_train_full.txt

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:177.)
  output = torch._nested_tensor_from_mask(
Using device: cuda
Loading data from data/porto_r0.1...
  Loading preprocessor state from checkpoints/pretrain/preprocessor.pkl
  Grids: 6156  Segments: 11491
  ATA-Graph edges: 59528
  Split: train=5600  val=800  test=1600
Loading pretrained encoder from checkpoints/pretrain/best_encoder.pt...
  Pretrained encoder loaded successfully.
Model parameters: 5,795,200

Starting supervised training for 100 epochs...
Epoch    1/100  train_loss=8.8890  val_loss=8.4546  P=0.0031  R=0.0031  F1=0.0031  lr=5.00e

In [4]:
!cp hstg_*_full.txt /content/drive/MyDrive/cm4604_cp3/
!cp -r HSTGMatch/checkpoints /content/drive/MyDrive/cm4604_cp3/hstg_checkpoints

**Decision rule for Job B:** watch the `F1=` value in the training log. If it is still below about 0.05 after roughly 30 epochs, stop the run (Runtime > Interrupt) and report honestly that HSTGMatch could not be reproduced at our data scale. Do not spend more than this one session on it.